In [1]:

import json
import pennylane as qml
import pennylane.numpy as np
import scipy

H = np.array([[ 0.39488016,  0.04722628, -0.17943126, -0.03673282],
        [ 0.04722628,  0.37758558,  0.12997088,  0.17848188],
        [-0.17943126,  0.12997088,  0.53582574,  0.01115543],
        [-0.03673282,  0.17848188,  0.01115543,  0.29170851]])

U = scipy.linalg.expm(2j * np.pi * H)

def state_prep(params, wires):
    """
    Implements the state preparation circuit.

    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the circuit.
        - wires (list): Labels for the circuit wires.
    Returns:
        - Does not return anything since it is a subcircuit.
    """
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[2], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])

dev = qml.device('lightning.qubit', wires = range(8))

@qml.qnode(dev)
def qpe_circuit(params):
    """
    Implements the QPE routine for the Unitary U, including initial state 
    preparation using the state_prep subcircuit.

    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the state_prep circuit.
    Returns:
        - np.tensor(float): Computational basis probabilities in the estimation wires.
    """
    estimation_wires = list(range(6))
    target_wires = [6,7]

    # State preparation on target wires
    state_prep(params, wires=target_wires)

    # Prepare estimation wires in |+>
    for w in estimation_wires:
        qml.Hadamard(wires=w)

    # Apply controlled-U^{2^k} operations for k=0...5
    # QPE convention: wire k controls U^{2^k}
    for k, ctl_wire in enumerate(estimation_wires):
        power = 2 ** k
        # Calculate U^{2^k}
        Uk = np.linalg.matrix_power(U, power)
        qml.ctrl(qml.QubitUnitary, control=ctl_wire)(Uk, wires=target_wires)

    # Inverse QFT on estimation wires
    qml.adjoint(qml.QFT)(wires=estimation_wires)

    # Measure estimation register in computational basis
    return qml.probs(wires=estimation_wires)

def compute_statistics(params):
    """
    Computes the phase and its uncertainty by postprocessing the results of the QPE circuit.

    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the state_prep circuit.
    Returns:
        - mu (float): The phase calculated as a weighted average.
        - sigma (float): The uncertainty calculated as the standard deviation
        for the phase.
    """
    probs = qpe_circuit(params)  # shape: (64,)

    # The output index is bit-reversed; correct phases assignment
    num_bits = 6
    n = 2 ** num_bits
    # For i from 0 to 63, its bit-reversed index is int(format(i, '06b')[::-1], 2); associate phase bit_reverse(i)/n
    bitreversed_indices = np.array([int(format(i, "06b")[::-1], 2) for i in range(n)])
    phases = bitreversed_indices / n

    mu = float(np.dot(probs, phases))
    sigma = float(np.sqrt(np.dot(probs, (phases - mu) ** 2)))
    return mu, sigma

# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    outs = list(compute_statistics(ins))
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, atol = 1e-4)

# These are the public test cases
test_cases = [
    ('[1.35889209, -0.6219561, -1.31577162]', '[0.121662, 0.120539]'),
    ('[4.40084815, -0.77288063,  0.6425846]', '[0.303264, 0.043692]')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
        except AssertionError:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
        else:
            print("Correct!")


Running test case 0 with input '[1.35889209, -0.6219561, -1.31577162]'...
Wrong Answer. Have: '[0.4493544504475859, 0.2653901081081632]'. Want: '[0.121662, 0.120539]'.
Running test case 1 with input '[4.40084815, -0.77288063,  0.6425846]'...
Wrong Answer. Have: '[0.359956370667439, 0.288266525699648]'. Want: '[0.303264, 0.043692]'.


/home/ma8183/anaconda3/envs/env/lib/python3.11/site-packages/pennylane/ops/op_math/decompositions/two_qubit_unitary.py:621: RuntimeWarning: The two-qubit decomposition may not be differentiable when the input unitary depends on trainable parameters.
  _check_differentiability_warning(U)
